# `nb11`: Estimating air pollution from satellite data

This notebook is Julien Gustin's solution for a past homework assignment in the course. 

It demonstrates the application of Bayesian modeling techniques to a real-world dataset, focusing on the relationship between aerosol optical depth and PM2.5 concentration. The notebook includes data preprocessing, model fitting, and posterior predictive checks, all while adhering to best practices in data visualization and scientific computing.

# Introduction

Air pollution is a major threat to global health, with 3 million deaths each year linked to ambient fine particulate matter pollution (PM2.5). While ground monitoring networks provide information on PM2.5 concentrations, there are still many regions with limited coverage, particularly in low-income areas of Africa or central Asia.

![](./figures/nb11/particulate-matter.png)

To better understand the public health effects of PM2.5, we need to estimate its concentration at a high spatial resolution. To do this, we will use a combination of direct measurements from ground monitors and satellite data. The satellite data provides high-resolution measurements of aerosol optical depth, which we can convert into estimates of PM2.5 concentration. By calibrating the satellite data using the ground monitor measurements, we can obtain estimates of PM2.5 concentration at the required spatial resolution.

In this case study, our goal will be to develop a predictive model of PM2.5 concentration using satellite data, with properly calibrated prediction intervals.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
from sklearn.cluster import KMeans
from scipy.stats import norm
from scipy.optimize import minimize
import corner
import emcee
from emcee import autocorr
from multiprocessing import Pool
from utils import * # most of the functions related to plotting are in this file
import os

os.environ["OMP_NUM_THREADS"] = "1"

# Data exploration

<div class="alert alert-success">
    
**Exercise**. Load the dataset `"pm25.csv"` into a Pandas DataFrame. Explore the dataset and report relevant insights.
    
</div>

In [ ]:
df = pd.read_csv('pm25.csv', sep=',')
df = df.set_index('Unnamed: 0')
df.index.name = 'index'
df = df.sort_values(by=['super_region'])
df = df.reset_index(drop=True)
# log transformation 
df['log_pm25'] = np.log(df['pm25'])
df['log_aod'] = np.log(df['sat_2014'])
# to start at 0 instead of 1
df['super_region'] = df['super_region'] - 1 

In [ ]:
df.head(10)

The dataset contains 4 qualitative variables
- **City_locality**: name of the city
- **iso3**: ISO 3166-1 alpha-3 code of the country
- **country**: name of the country
- **super_region_name**: name of the super region, the super regions are based on two criteria: epidemiological similarity and geographic proximity. [Source](https://www.iapb.org/fr/learn/vision-atlas/about/definitions-and-regions/)
  - *HighIncome*
  - *Sub-Saharan Afr*
  - *E-Eur/C-Eur/C-Asia*
  - *N-Afr/MidEast*
  - *S-Asia*
  - *LatAm/Carib*
  - *SE-Asia/E-Asia/Oceania*
  

and 3 quantitative variables
- **pm25**: PM2.5 concentration measured at the ground monitor (in $μg/m^3$)
- **sat_2014**: aerosol optical depth measurements from satellite (AOD)
- **super_region**: super region code [0-6]
  - 0: *HighIncome*
  - 1: *Sub-Saharan Afr*
  - 2: *E-Eur/C-Eur/C-Asia*
  - 3: *N-Afr/MidEast*
  - 4: *S-Asia*
  - 5: *LatAm/Carib*
  - 6: *SE-Asia/E-Asia/Oceania*

**Note** *super_region* and *super_region_name* represent the same things but with different representations. 

In [ ]:
df.describe()

- The median and mean for both variables are very close to each other. This indicates that the distributions are not too skewed, with not many outliers. 
- The difference between the third quartile and median compared to the first quartile and median is slight but may indicate that the distribution is not symmetrical for the two variables.
  - *pm25*
    -  50% - 25%: $14.5 - 9.5 = 5$   $μg/m^3$
    -  75% - 50%: $24 - 14.5 = 9.5$  $μg/m^3$
  - *sat_2014*
    - 50% - 25%: $15.5 - 9.68 = 5.82$ $AOD$
    - 75% - 50%: $22.2 - 15.5 = 6.7$ $AOD$
- Finally it is worth noticing that the two variable have relatively the same scale


In [ ]:
df[df.isna().any(axis=1)]

The dataset contains 3 NaN for the variable *city_locality*, only. This is not a problem as this variable is not used in the analysis.

In [ ]:
# check for each country the number of different super_region
super_region_unique = df.groupby('country').super_region.nunique() 
super_region_unique[super_region_unique > 1]

*Bosnie and Herzegovina* is the only country with 2 different super_region. Other countries have only one super_region.

In [ ]:
plt.figure(figsize=(20, 15))

plt.subplot(2, 1, 1)
plt.title('Number of observations per super_region_name')
plt.xlabel('super_region_name')
plt.ylabel('Number of observations')
df['super_region_name'].value_counts().plot(kind='bar')

plt.subplots_adjust(hspace=0.75)

plt.subplot(2, 1, 2)
plt.title('Number of observations per country')
plt.xlabel('Country')
plt.ylabel('Number of observations')
df['country'].value_counts().plot(kind='bar')
plt.show()

The above plot show that the dataset is mainly composed of data from *High-income* countries such as USA, France, Italy and Spain while other countries are underrepresented. 

In [ ]:
sns.set(style="whitegrid")
plt.figure(figsize=(15,5))
sns.boxplot(x="super_region_name", y="pm25", data=df)
plt.title("PM2.5 distribution by super region")
plt.show()

sns.set(style="whitegrid")
plt.figure(figsize=(15,5))
sns.boxplot(x="super_region_name", y="sat_2014", data=df)
plt.title("AOD distribution by super region ")
plt.show()

The distribution of *sat_2014* and *pm25* are very similar for the different super region. This reinforces the idea that the *sat_2014* variable may be a good predictor for the *pm25* variable.
Also if the variable *pm25* would be independent of the *super_region* variable, we would expect the distribution of *pm25* to be the same for each super region. This is not the case therefore *super_region* variable is not independent of the *pm25* variable. (same for *sat_2014*) However, it may also come from the fact that the dataset is not balanced for each super region.

<div class="alert alert-success">
    
**Exercise**. Plot and compare the ground measurements of PM2.5 concentration (column `pm25`, in $μg/m^3$) against the  aerosol optical depth satellite measurements (column `sat_2014`). Discuss your observations.
    
Tips: Look at the data in different ways.
    
</div>

In what follows, I will study the data on a logarithmic scale for better visualization and because later the logarithmic transformation of the data will be use in the model.

In [ ]:
plt.figure(figsize=(20,12))
plt.subplot(2, 1, 1)
plt.scatter(df['sat_2014'], df['pm25'], alpha=0.5)
plt.title("PM2.5 concentration against the  aerosol optical depth satellite measurements")
# draw a fitted line
sns.regplot(x="sat_2014", y="pm25", data=df, scatter=False, color="r", ci=None)
plt.xlabel(r"Aerosol optical depth satellite measurements - $log(AOD)$")
plt.ylabel(r"PM2.5 concentration - $log(\mu g/m^3)$")
# add legend
plt.legend(labels=['Data', 'Fitted line'])

plt.subplot(2, 1, 2)
# space
plt.subplots_adjust(hspace=0.3)
plt.scatter(df['log_aod'], df['log_pm25'], alpha=0.5)
plt.title("PM2.5 concentration against the  aerosol optical depth satellite measurements in log scale")
# draw a fitted line
sns.regplot(x=df['log_aod'], y=df['log_pm25'], data=df, scatter=False, color="r", ci=None)
plt.xlabel(r"Aerosol optical depth satellite measurements - $log(AOD)$")
plt.ylabel(r"PM2.5 concentration - $log(\mu g/m^3)$")

plt.legend(labels=['Data', 'Fitted line'])

plt.show()

In [ ]:
# compute correlation coefficient
df[['log_aod', 'log_pm25']].corr()

The plot suggests a linear relationship between the two variables, which is supported by the high correlation coefficient of 0.77. However, there is still some residual noise present in the data.

In [ ]:
X = df['log_aod']
y = df['log_pm25']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit() # Ordinary Least Squares, assume normal distribution of the noise
predictions = model.predict(X)

# plot histogram of the residuals
plt.figure(figsize=(15,6))
plt.hist(model.resid, bins=50)
plt.title("Histogram of the residuals between a line fitted via Ordinary Least Squares and the data")
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.show()

In [ ]:
from scipy.stats import kurtosis, skew
print("Kurtosis: ", kurtosis(model.resid))
print("Skewness: ", skew(model.resid))

Under the normality assumptions, we expect the residuals between OLS and the data to be normal and thus the skewness to be 0 and the kurtosis to be 3. However, in this case, the kurtosis is 2.0 and the skewness is 0.7, suggesting that the distribution of residuals is not normal. Therefore, it is important to keep in mind that models based on the normality assumption of the output given the input may not be appropriate. However, the distribution of residuals is not too far from a gaussian, so we can expect the model to still fit the data well.

Let's split the dataset by *super_region* to see if the relationship between *pm25* and *sat_2014* is the same for each super region.

In [ ]:
scatter_plot_all_clusters(df, 'super_region_name', 'PM2.5 concentration against the  aerosol optical depth satellite measurements by super region')

In [ ]:
scatter_plot_per_cluster(df, 'super_region', 'super_region_name', 'PM2.5 concentration against the  aerosol optical depth satellite measurements with respect to super region', True, True)

The graphs above show that the relationship between *pm25* and *sat_2014* is not the same for each super region. For example, for *high income* countries, the correlation coefficient is $0.67$, while for *E-Eur/C-Eur/C-Asia* countries, the correlation coefficient is $0.3$. This indicates that having a linear model for each super region may not be the best choice. But also reinforces the idea that a different model may be needed for each super region.

We can notice a nice linear relationship between *pm25* and *sat_2014* for the super regions *High-income*, *SE-Asia/E-Asia/Oceania*, and *S-Asia*. In these cases, the use of a linear model is relevant.

In [ ]:
plt.figure(figsize=(15,12))

plt.subplot(2, 1, 1)
plt.title("Distribution of measurements of the aerosol optical depth satellite in log scale")
plt.hist(df['log_aod'], bins=30, color='blue', alpha=0.5)
plt.xlabel(r"$log(AOD)$")
plt.ylabel("Frequency")
# share x
plt.subplot(2, 1, 2, sharex=plt.gca())
plt.title("Distribution of measurements of the PM2.5 concentration in log scale")
plt.hist(df['log_pm25'], bins=30, color='orange', alpha=0.5)
plt.xlabel(r"$log(\mu g/m^3)$")
plt.ylabel("Frequency")
plt.show()

The distribution of the two variable in log scale are slightly different.

<div class="alert alert-success">
    
**Exercise**. Identify $K=7$ groups of countries by clustering based on ground measurements. Discuss any differences with the World Health Organizations super-regions (column `super_region`).

Tips: Aggregate measurements by countries and then cluster countries using `sklearn.cluster.KMeans`.
    
</div>

In [ ]:
df_country = df.groupby('country').agg({'log_pm25': 'mean', 'super_region': 'first', 'super_region_name': 'first', 'country': 'first'})
df_country.rename(columns={'log_pm25': 'log_pm25_mean'}, inplace=True)
# sort df_country by super_region
df_country.sort_values(by=['super_region'], inplace=True)
df_country.reset_index(drop=True, inplace=True)
# same for df
df.sort_values(by=['super_region'], inplace=True)
df.reset_index(drop=True, inplace=True)

X = df_country['log_pm25_mean'].to_numpy().reshape(-1, 1)
kmeans = KMeans(n_clusters=7, random_state=42).fit(X)
labels = kmeans.labels_

centroids = kmeans.cluster_centers_

# merge the cluster labels with the original data
df_country['cluster'] = labels
df_country['cluster_name'] = df_country['cluster'].map({0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2', 3: 'Cluster 3', 4: 'Cluster 4', 5: 'Cluster 5', 6: 'Cluster 6'})
df = df_country.merge(df, on='country', how='left', suffixes=('', '_y'))
df.drop(columns=['log_pm25_mean', 'super_region_y', 'super_region_name_y'], inplace=True)

In [ ]:
plot_clusters(df_country, 'log_pm25_mean', "Clustering of groups of countries based on ground measurements and by super-regions | Country level")

Examining the relationship between *PM2.5* concentration and *super region*, we observe significant overlap between the various super regions which will complicate clustering. This is not surprising, as the definition of super regions is not based on *PM2.5* concentration but based on two criteria: epidemiological similarity and geographic closeness. However, we do notice that countries labeled as *HighIncome* super regions tend to have relatively close *PM2.5* values, which can lead to distinct clusters compared to other super regions.

In [ ]:
plot_clusters(df, 'log_pm25', "Clustering of groups of countries based on ground measurements and by super-regions | City level")

In [ ]:
plt.figure(figsize=(15,10))

for i in range(7):
    plt.subplot(2, 4, i+1)
    plt.title("Cluster " + str(i))
    plt.ylabel("Number of observations")
    df[df.index.isin(df[df["cluster"] == i].index)]['super_region_name'].value_counts().plot(kind='bar')

    plt.gca().get_children()[np.argmax(df[df.index.isin(df["cluster"].index)]['super_region_name'].value_counts())].set_color('r')
    plt.gca().get_xticklabels()[np.argmax(df[df.index.isin(df["cluster"].index)]['super_region_name'].value_counts())].set_fontweight('bold')

plt.tight_layout()
plt.suptitle("Number of different super regions contained in each cluster", fontsize=16)
plt.subplots_adjust(top=0.92)

plt.show()

As expected, the clusters are not well-defined with respect to super regions. This is due to the fact that the super regions are not determined by *PM2.5* concentration, but rather by epidemiological similarity and geographic proximity. However, K-means is able to effectively distinguish between the *HighIncome* super region compare to the other super regions

In [ ]:
scatter_plot_all_clusters(df.sort_values(by=['cluster']), 'cluster_name', 'PM2.5 concentration against the  aerosol optical depth satellite measurements by cluster')

In [ ]:
scatter_plot_per_cluster(df.sort_values(by=['cluster']), 'cluster', 'cluster_name', 'PM2.5 concentration against the  aerosol optical depth satellite measurements with respect to cluster', True, True)

When comparing these correlations with the one computed earlier using *super_region* as cluster, we observe that in the correlation coefficent is lower, which will certainly affect the model performance.

# Build: Probabilistic modelling

In statistical terms, our goal is to build a probabilistic model $p(\log \text{PM}_{2.5} | \log \text{AOD})$ of PM2.5 concentration, where $\log \text{PM}_{2.5}$ are ground measurements (`log(pm25)`) and $\log \text{AOD}$ are aerosol optical depth measurements from a satellite (`log(sat_2014)`).

Throughout this case study, we will build and evaluate 3 distinct probabilistic models. The first model will assume that satellite measurements are a good predictor of the ground measurements. The second and third models will try to account for the geographical and socio-demographic heterogeneity between countries. Using $y$ to denote log-measurements of PM2.5 concentration and $x$ to denote log-measurements taken by satellite, we define:
- Model 1: a linear regression model $y_i \sim \mathcal{N}(\beta_0 + \beta_1 x_i, \sigma^2)$, where $i$ ranges over all observations.
- Model 2: a hierarchical linear regression model stratified by super-region $y_{ij} \sim \mathcal{N}((\beta_0 + \beta_{0j}) + (\beta_1 + \beta_{1j}) x_{ij}, \sigma^2), \beta_{0j} \sim \mathcal{N}(0, \tau_0^2), \beta_{1j} \sim \mathcal{N}(0, \tau_1^2)$, where $i$ ranges over the observations in each super-region and $j$ ranges over the super-regions.
- Model 3: a hierarchical linear regression model stratified by cluster $y_{ij} \sim \mathcal{N}((\beta_0 + \beta_{0j}) + (\beta_1 + \beta_{1j}) x_{ij}, \sigma^2), \beta_{0j} \sim \mathcal{N}(0, \tau_0^2), \beta_{1j} \sim \mathcal{N}(0, \tau_1^2)$, where $i$ ranges over the observations in each cluster and $j$ ranges over the clusters.

Super-regions in Model 2 correspond to the World Health Organizations super-regions (column `super_region`), while clusters in Model 3 correspond to groups of countries found by clustering based on ground measurements of PM2.5 (see above). Model parameters $\sigma, \beta_0, \beta_1, \tau_0, \tau_1$ need prior distributions.

In [ ]:
def plot_model(N, prior_model, model, ylim): 
    X = np.linspace(0, 5, 1000)
    plt.figure(figsize=(12,6))

    for i in range(N):
        Y = model(X, prior_model, True)
        plt.scatter(X, Y, alpha=0.1)
        

    plt.title("Prior distribution of the model parameters")
    plt.xlabel("sat_2014 (aerosol optical depth satellite measurements)")
    plt.ylabel("pm2.5 concentration (ug/m3)")
    
    # set ylim
    if ylim is not None:
        plt.ylim(*ylim)

    plt.show()

def plot_model_cluster(N, prior_model, model, ylim): 
    X = np.linspace(0, 5, 1000)
    J = np.random.randint(0, 7, len(X))

    colors = ['r', 'g', 'b', 'c', 'm', 'y', 'k']

    plt.figure(figsize=(12,6))

    for _ in range(100):
        Y = model(X, J, prior_model, True)
        plt.scatter(X, Y, alpha=0.1)
        

    plt.title("Prior distribution of the model parameters")
    plt.xlabel("sat_2014 (aerosol optical depth satellite measurements)")
    plt.ylabel("pm2.5 concentration (ug/m3)")

    if ylim is not None:
        plt.ylim(*ylim)
    plt.show()

<div class="alert alert-success">
    
**Exercise**. Implement Model 1, 2 and 3 defined above. First consider an uninformative prior for the model parameters.
    
</div>

In [ ]:
def prior_model_1_uninformative():
    b0 = np.random.uniform(-1e6, 1e6)
    b1 = np.random.uniform(-1e6, 1e6)
    sigma = np.exp(np.random.normal(0, 3))
    return b0, b1, sigma

def model_1(x, sampler, sample_from_prior:bool):
    b0, b1, sigma = sampler()
    return np.random.normal(b0 + b1*x, sigma)

plot_model(100, prior_model_1_uninformative, model_1, ylim=None)

In [ ]:
def prior_model_2_uninformative():
    b0 = np.random.uniform(-1e6, 1e6)
    b1 = np.random.uniform(-1e6, 1e6)
    sigma = np.exp(np.random.normal(0, 3))
    tau_0 = np.exp(np.random.normal(0, 3))
    tau_1 = np.exp(np.random.normal(0, 3))
    return b0, b1, sigma, tau_0, tau_1

def model_2(X, J, sampler, sample_from_prior:bool):
    if sample_from_prior:
        b0, b1, sigma, tau_0, tau_1 = sampler()
        b0js = norm(0, tau_0).rvs(7)
        b1js = norm(0, tau_1).rvs(7)
    else:
        b0, b1, sigma, tau_0, tau_1, b0js, b1js = sampler()

    J = np.array(J).astype(int)
    Y = np.random.normal((b0 + b0js[J]) + (b1 + b1js[J]) * X, sigma)
    
    return Y

plot_model_cluster(100, prior_model_2_uninformative, model_2, ylim=None)

In [ ]:
def prior_model_3_uninformative():
    b0 = np.random.uniform(-1e6, 1e6)
    b1 = np.random.uniform(-1e6, 1e6)
    sigma = np.exp(np.random.normal(0, 3))
    tau_0 = np.exp(np.random.normal(0, 3))
    tau_1 = np.exp(np.random.normal(0, 3))
    return b0, b1, sigma, tau_0, tau_1

def model_3(X, J, sampler, sample_from_prior:bool):
    if sample_from_prior:
        b0, b1, sigma, tau_0, tau_1 = sampler()
        b0js = norm(0, tau_0).rvs(7)
        b1js = norm(0, tau_1).rvs(7)
    else:
        b0, b1, sigma, tau_0, tau_1, b0js, b1js = sampler()

    J = np.array(J).astype(int)
    Y = np.random.normal((b0 + b0js[J]) + (b1 + b1js[J]) * X, sigma)
    
    return Y

plot_model_cluster(100, prior_model_3_uninformative, model_3, ylim=None)

<div class="alert alert-success">
    
**Exercise**. Using a prior predictive check, evaluate and discuss if the prior predictive distribution is realistic. Propose a weakly informative prior that results in more realistic simulated data.
    
Tips: Look for PM2.5 typical concentrations in polluted areas to ballpark a reasonable upper bound on realistic realizations.
    
</div>

It goes from 5.9 up to 100 $μg/m^3$, in log scale it goes from 1.77 to 4.6 $log(μg/m^3)$. Therefore in this prior predictive check I will evaluate my prior predictive distribution between with respect to the minimum and maximum values of the log scale of the PM2.5 concentration.

Source: https://databank.worldbank.org/source/world-development-indicators/Series/EN.ATM.PM25.MC.M3

As I am not in expert in the field I will use prior that are weakly informative for the 3 models.
After doing some research I have found the following information:
- It seems to have a positive linear relation ship between the two variables
- The maximum $log(pm2.5)$ observed is 4.6 $log(μg/m^3)$
- The minimum $log(pm2.5)$ observed is 1.77 $log(μg/m^3)$

This allow me to define the prior in order to have something realistic. However, as I do not have knowledge of the subject I will use distributions that don't have a fixed interval.

### Model 1

In [ ]:
def plot_min_max(Y):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12,6))

    ax1.hist(np.min(Y, axis=1))
    ax1.set_title("Min distribution")
    ax2.set_xlabel(r"pm2.5 concentration - $\log(\mu g/m^3)$")
    # add hline
    ax1.axvline(1.77, color='r', linestyle='dashed', linewidth=1)
    ax1.set_ylabel("frequency")
    # add space 
    plt.subplots_adjust(hspace=0.5)

    ax2.hist(np.max(Y, axis=1))
    ax2.set_title("Max distribution")
    ax2.axvline(4.2, color='r', linestyle='dashed', linewidth=1)
    ax2.set_xlabel(r"pm2.5 concentration - $\log(\mu g/m^3)$")
    ax2.set_ylabel("frequency")

    plt.show()


In [ ]:
X = np.linspace(0, 5, 1000)

Y = []
for i in range(100):
    Y.append(model_1(X, prior_model_1_uninformative, True))

Y = np.array(Y)
plot_min_max(Y)

PM2.5 values are in a too wide range to be realistic.

In [ ]:
def prior_model_1_weakly_informative():
    b0 = np.random.normal(0, 2)
    b1 = np.random.normal(1, 2)
    sigma = np.exp(np.random.normal(0, 1))
    return b0, b1, sigma

plot_model(100, prior_model_1_weakly_informative, model_1, ylim=(-5, 10))

In [ ]:
X = np.linspace(0, 5, 1000)

Y = []
for i in range(100):
    Y.append(model_1(X, prior_model_1_weakly_informative, True))

Y = np.array(Y)
plot_min_max(Y)

The range is much better, still too wide but it is ok with respect to my knowledge of the field.

### Model 2

In [ ]:
X = np.linspace(0, 5, 1000)
J = np.random.randint(0, 7, len(X))

Y = []
for _ in range(100):
    Y.append(model_2(X, J, prior_model_2_uninformative, True))

Y = np.array(Y)    
plot_min_max(Y)

PM2.5 values are in a too wide range to be realistic.

In [ ]:
def prior_model_2_weakly_informative():
    b0 = np.random.normal(0, 3)
    b1 = np.random.normal(1, 2)
    sigma = np.exp(np.random.normal(0, 1))
    tau_0 = np.exp(np.random.normal(0, 1))
    tau_1 = np.exp(np.random.normal(0, 1))
    return b0, b1, sigma, tau_0, tau_1

plot_model_cluster(100, prior_model_2_weakly_informative, model_2, ylim=(-5, 10))

In [ ]:
X = np.linspace(0, 5, 1000)
J = np.random.randint(0, 7, len(X))

Y = []
for _ in range(100):
    Y.append(model_2(X, J, prior_model_2_weakly_informative, True))

Y = np.array(Y)    
plot_min_max(Y)

The range is much better, still too wide but it is ok with respect to my knowledge of the field.

### Model 3

In [ ]:
X = np.linspace(0, 5, 1000)
J = np.random.randint(0, 7, len(X))

Y = []
for _ in range(100):
    Y.append(model_3(X, J, prior_model_3_uninformative, True))

Y = np.array(Y)    
plot_min_max(Y)

PM2.5 values are in a too wide range to be realistic.

In [ ]:
def prior_model_3_weakly_informative():
    b0 = np.random.normal(0, 3)
    b1 = np.random.normal(1, 2)
    sigma = np.exp(np.random.normal(0, 1))
    tau_0 = np.exp(np.random.normal(0, 1))
    tau_1 = np.exp(np.random.normal(0, 1))
    return b0, b1, sigma, tau_0, tau_1

plot_model_cluster(100, prior_model_3_weakly_informative, model_3, ylim=(-5, 10))

In [ ]:
X = np.linspace(0, 5, 1000)
J = np.random.randint(0, 7, len(X))

Y = []
for _ in range(100):
    Y.append(model_3(X, J, prior_model_3_weakly_informative, True))

Y = np.array(Y)    
plot_min_max(Y)

The range is much better, still too wide but it is ok with respect to my knowledge of the field.

# Compute: Posterior inference

Repeat the experiments below for each of the three models:

### Model 1

<div class="alert alert-success">
    
**Exercise**. Compute the posterior distribution of all unobserved random variables.
    
Tips: If you use MCMC, initialize the chain(s) around the MAP estimate.
    
</div>

In [ ]:
df_1 = df.copy()

def log_prior_model_1(b0, b1, sigma):
    return norm.logpdf(b0, 0, 2) + norm.logpdf(b1, 1, 2) + norm.logpdf(np.log(sigma), 0, 1)

def log_likelihood_model_1(x, y, b0, b1, sigma):
    return norm.logpdf(y, b0 + b1*x, sigma).sum()

def log_posterior_model_1(theta):
    x, y = df_1['log_aod'], df_1['log_pm25']
    b0, b1, log_sigma = theta
    sigma = np.exp(log_sigma)
    lp = log_prior_model_1(b0, b1, sigma)
    ll = log_likelihood_model_1(x, y, b0, b1, sigma)
    
    return lp + ll

In [ ]:
nll = lambda *args: -log_posterior_model_1(*args)

# random initial guess
theta_0 = prior_model_1_weakly_informative()

res1 = minimize(nll, theta_0, options={"disp":True})
b0, b1, log_sigma = res1.x
sigma = np.exp(log_sigma)

In [ ]:
nwalkers = 50
ndim = 3
nsteps = 5000
burnin = 1000

labels_1 = ["b0", "b1", "sigma"] 
init_states = res1.x + 1e-3 * np.random.randn(nwalkers, ndim)

In [ ]:
with Pool() as pool:
    sampler_1 = emcee.EnsembleSampler(nwalkers, ndim, log_posterior_model_1, pool=pool)
    sampler_1.run_mcmc(init_states, nsteps, progress=True)

<div class="alert alert-success">
    
**Exercise**. Assess whether your inference results are faithful.
    
</div>

Let's look at the chains first.

In [ ]:
plot_walkers(sampler_1, ndim=ndim, labels=labels_1, discard=burnin)

- The chains look like random walks and do not show any clear trend (after discarding the first 1000 iterations), which is a good sign tjat the chains may have converged.

In [ ]:
tau = sampler_1.get_autocorr_time(quiet=True)
thin = int(np.max(tau))
print(f"burnin: {burnin}, thin: {thin}")

thetas_1 = sampler_1.get_chain(flat=True, discard=burnin, thin=thin)

- The auto correlation time reaches a maximum value of 38, which is not excessively high and reinforces the fact that we may have reached convergence.

In [ ]:
fig = corner.corner(thetas_1, labels=labels_1, truths=res1.x) # truths represent here the MAP estimate

- The posterior distribution of the parameters is centered around the maximum a posteriori (MAP) estimate, which is a good indication because the MAP estimate the mode of the posterior distribution.

<div class="alert alert-success">
    
**Exercise**. Plot the posterior distributions. 
    
</div>

In [ ]:
plt.figure(figsize=(15,6))
plt.scatter(df['log_aod'], df['log_pm25'], alpha=0.5, label='data')
plt.title("Posterior distribution of the model parameters | Model 1")
x0 = np.linspace(0, 5, 1000) 

# Compute the MAP estimate
y0 = b0 + b1*x0

# Plot the posterior distribution
for i, theta in enumerate(thetas_1[:1000]):
    if i == 0:
        plt.plot(x0, np.dot(np.vander(x0, 2), np.array([theta[1], theta[0]])), "r", alpha=0.5, label='Linear predictors posterior distribution')
    plt.plot(x0, np.dot(np.vander(x0, 2), np.array([theta[1], theta[0]])), "r", alpha=0.05)

plt.plot(x0, y0, '--', label=r'MAP', color='orange')

plt.xlabel(r"Aerosol optical depth satellite measurements - $\log(AOD)$")
plt.ylabel(r"PM2.5 concentration -$\log(\mu g/m^3)$")

plt.legend()
plt.show()    

### Model 2

<div class="alert alert-success">
    
**Exercise**. Compute the posterior distribution of all unobserved random variables. 
    
Tips: If you use MCMC, initialize the chain(s) around the MAP estimate.
    
</div>

In [ ]:
df_2 = df.copy()

def log_prior_model_2(b0, b1, sigma, tau_0, tau_1, bj0s, bj1s):
    return norm.logpdf(b0, 0, 3) + norm.logpdf(b1, 1, 2) + norm.logpdf(np.log(sigma), 0, 1) + norm.logpdf(np.log(tau_0), 0, 1) + norm.logpdf(np.log(tau_1), 0, 1)\
            + norm.logpdf(bj0s, 0, tau_0).sum() + norm.logpdf(bj1s, 0, tau_1).sum()

def log_likelihood_model_2(X, Y, J, b0, b1, sigma, bj0s, bj1s):
    ll = norm.logpdf(Y, (b0 + bj0s[J]) + (b1 + bj1s[J]) * X, sigma).sum()
        
    return ll

def log_posterior_model_2(theta):
    # for optimization purpose `df` is global here https://emcee.readthedocs.io/en/stable/tutorials/parallel/
    X, Y, J = df_2['log_aod'], df_2['log_pm25'], df_2['super_region']
    b0, b1, log_sigma, log_tau_0, log_tau_1, *bjs = theta
    bj0s = np.array(bjs[:7])
    bj1s = np.array(bjs[7:])
    sigma = np.exp(log_sigma)
    tau_0 = np.exp(log_tau_0)
    tau_1 = np.exp(log_tau_1)
    lp = log_prior_model_2(b0, b1, sigma, tau_0, tau_1, bj0s, bj1s)
    ll = log_likelihood_model_2(X, Y, J, b0, b1, sigma, bj0s, bj1s)
    
    return lp + ll

In [ ]:
nll = lambda *args: -log_posterior_model_2(*args)

# random initial guess
theta_0 = np.array([0.1]*(7+7+2+1+2) * np.random.randn(19))

res2 = minimize(nll, theta_0, options={"disp":True})
b0, b1, log_sigma, log_tau_0, log_tau_1, *bjs = res2.x
bj0s = bjs[:7]
bj1s = bjs[7:]
sigma = np.exp(log_sigma)
tau_0 = np.exp(log_tau_0)
tau_1 = np.exp(log_tau_1)

In [ ]:
nwalkers = 300
ndim = 19
nsteps = 30000
burnin = 5000
labels_2 = [r"b0", r"b1", r"log sigma", r"log tau_0", r"log tau_1"] + [r"bj0_1", r"bj0_2", r"bj0_3", r"bj0_4", r"bj0_5", r"bj0_6", r"bj0_7"] + [r"bj1_1", r"bj1_2", r"bj1_3", r"bj1_4", r"bj1_5", r"bj1_6", r"bj1_7"]

In [ ]:
init_states = res2.x + 1e-2 * np.random.randn(nwalkers, ndim)

run_backend = emcee.backends.HDFBackend("sampler_2.h5")
run_backend.reset(nwalkers, ndim)
with Pool() as pool:
    sampler_2 = emcee.EnsembleSampler(nwalkers, ndim, log_posterior_model_2, pool=pool, backend=run_backend)
    sampler_2.run_mcmc(init_states, nsteps, progress=True)

In [ ]:
# sampler_2 = emcee.backends.HDFBackend("sampler_2.h5")

<div class="alert alert-success">
    
**Exercise**. Assess whether your inference results are faithful.
    
</div>

In [ ]:
plot_walkers(sampler_2, ndim=ndim, labels=labels_2, discard=burnin)

- The chains look like random walks and do not show any clear trend (after discarding the first 5000 iterations), which is a good sign that the chains may have converged. 
- However, it is worth noting that MCMC needs way more walkers and iterations to converge than the first model.

In [ ]:
tau = sampler_2.get_autocorr_time(quiet=True)
thin = int(np.max(tau))
print(f"burnin: {burnin}, thin: {thin}")
thetas_2 = sampler_2.get_chain(flat=True, discard=burnin, thin=thin)

- The auto correlation time reaches a maximum value of 293, which is higher than the first model, but still reasonable considering the number of walkers and iterations.

In [ ]:
fig = corner.corner(thetas_2, labels=labels_2)

- The posterior distribution of the parameters have the MAP estimate in high density region, which is a good indication because the MAP estimate the mode of the posterior distribution.
- However this result must be taken with caution because maybe that the MAP estimate is not the mode of the posterior distribution, but rather a local maximum and that mcmc has not converged yet but just stayed around the MAP estimate.

<div class="alert alert-success">
    
**Exercise**. Plot the posterior distributions. 
    
</div>

In [ ]:
plot_model_distribution(df, thetas_2[:1000], "Posterior distribution of the model parameters clustered by super region | Model 2", "super_region_name", "super_region")

In [ ]:
plot_model_distribution_per_cluster(df, thetas_2[:1000], "Posterior distribution of the model parameters clustered by super region in log scale | Model 2", "super_region_name")

### Model 3

<div class="alert alert-success">
    
**Exercise**. Compute the posterior distribution of all unobserved random variables. 
    
Tips: If you use MCMC, initialize the chain(s) around the MAP estimate.
    
</div>

In [ ]:
df_3 = df.copy()

def log_prior_model_3(b0, b1, sigma, tau_0, tau_1, bj0s, bj1s):
    return norm.logpdf(b0, 0, 3) + norm.logpdf(b1, 1, 2) + norm.logpdf(np.log(sigma), 0, 1) + norm.logpdf(np.log(tau_0), 0, 1) + norm.logpdf(np.log(tau_1), 0, 1)\
            + norm.logpdf(bj0s, 0, tau_0).sum() + norm.logpdf(bj1s, 0, tau_1).sum()

def log_likelihood_model_3(X, Y, J, b0, b1, sigma, bj0s, bj1s):
    ll = norm.logpdf(Y, (b0 + bj0s[J]) + (b1 + bj1s[J]) * X, sigma).sum()
        
    return ll

def log_posterior_model_3(theta):
    # for optimization purpose `df` is global here https://emcee.readthedocs.io/en/stable/tutorials/parallel/
    X, Y, J = df_3['log_aod'], df_3['log_pm25'], df_3['cluster']
    b0, b1, log_sigma, log_tau_0, log_tau_1, *bjs = theta
    bj0s = np.array(bjs[:7])
    bj1s = np.array(bjs[7:])
    sigma = np.exp(log_sigma)
    tau_0 = np.exp(log_tau_0)
    tau_1 = np.exp(log_tau_1)
    lp = log_prior_model_3(b0, b1, sigma, tau_0, tau_1, bj0s, bj1s)
    ll = log_likelihood_model_3(X, Y, J, b0, b1, sigma, bj0s, bj1s)
    
    return lp + ll

In [ ]:
nll = lambda *args: -log_posterior_model_3(*args)

# random initial guess
theta_0 = np.array([0.1] * (7+7+2+1+2) * np.random.randn(19))

res3 = minimize(nll, theta_0)
b0, b1, log_sigma, log_tau_0, log_tau_1, *bjs = res3.x
bj0s = bjs[:7]
bj1s = bjs[7:]
sigma = np.exp(log_sigma)
tau_0 = np.exp(log_tau_0)
tau_1 = np.exp(log_tau_1)

In [ ]:
nwalkers = 300
ndim = 19
nsteps = 25000
burnin = 5000
labels_3 = [r"b0", r"b1", r"log sigma", r"log tau_0", r"log tau_1"] + [r"bj0_1", r"bj0_2", r"bj0_3", r"bj0_4", r"bj0_5", r"bj0_6", r"bj0_7"] + [r"bj1_1", r"bj1_2", r"bj1_3", r"bj1_4", r"bj1_5", r"bj1_6", r"bj1_7"]

In [ ]:
init_states = res3.x + 1e-2 * np.random.randn(nwalkers, ndim)

# add backend
backend = emcee.backends.HDFBackend("sampler_3.h5")
backend.reset(nwalkers, ndim)

with Pool() as pool:
    sampler_3 = emcee.EnsembleSampler(nwalkers, ndim, log_posterior_model_3,  args=(df['log_aod'], df['log_pm25'], df['cluster']), pool=pool)
    sampler_3.run_mcmc(init_states, nsteps, progress=True, backend=backend)

In [ ]:
# sampler_3 = emcee.backends.HDFBackend("sampler_3.h5")

<div class="alert alert-success">
    
**Exercise**. Assess whether your inference results are faithful.
    
</div>

In [ ]:
plot_walkers(sampler_3, ndim=ndim, labels=labels_3, discard=burnin)

- The chains look like random walks and do not show any clear trend (after discarding the first 5000 iterations), which is a good sign that the chains may have converged. 
- However, it is worth noting that MCMC needs way more walkers and iterations to converge than the first model.

In [ ]:
tau = sampler_3.get_autocorr_time(quiet=True)
thin = int(np.max(tau))
print(f"burnin: {burnin}, thin: {thin}")

thetas_3 = sampler_3.get_chain(flat=True, discard=burnin, thin=thin)

- The auto correlation time reaches a maximum value of 289, which is higher than the first model, but still reasonable considering the number of walkers and iterations.

In [ ]:
fig = corner.corner(thetas_3, labels=labels_3) # truths represent here the MAP estimates 

- The posterior distribution of the parameters have the MAP estimate in high density region, which is a good indication because the MAP estimate the mode of the posterior distribution.

<div class="alert alert-success">
    
**Exercise**. Plot the posterior distributions. 
    
</div>

In [ ]:
plot_model_distribution(df.sort_values(by=['cluster']), thetas_3[:1000], "Posterior distribution of the model parameters clustered by super region | Model 3", "cluster_name", "cluster")

In [ ]:
plot_model_distribution_per_cluster(df.sort_values(by=['cluster']), thetas_3[:1000], "Posterior distribution of the model parameters clustered by $\log (pm2.5)$ | Model 3", "cluster_name")

# Criticize

<div class="alert alert-success">
    
**Exercise**. Compare the posterior predictive distributions of replicated data $\{y^\text{rep}\}$ for Model 1, 2 and 3. Discuss each in comparison to the distribution of the observed ground measurements.
    
</div>

### Model 1

In [ ]:
def generate_replicates_1(df, thetas):
    replicates = []
    for theta in thetas:
        def sampler_1():
            b0, b1, log_sigma = theta
            sigma = np.exp(log_sigma)
            return b0, b1, sigma
            
        replicate = model_1(df['log_aod'], sampler_1, sample_from_prior=False)
        replicates.append(replicate)
    return np.array(replicates)

In [ ]:
n_replicates = 4
b0 = thetas_1[:n_replicates, 0]
b1 = thetas_1[:n_replicates, 1]
sigma = np.exp(thetas_1[:n_replicates, 2])

title = r"Posterior predictive distributions of replicated data $\{y^{rep}\}$ for Model 1"
plot_posterior_predictive_distribution(thetas_1[:n_replicates], df, generate_replicates_1 , title, b0s=b0, b1s=b1, sigmas=sigma)

The first model is the simplest one defined as $y_i \sim \mathcal{N}(\beta_0 + \beta_1 x_i, \sigma^2)$, the above plot shows the posterior predictive distribution of four replicated data $\{y^{\text{rep}}\}$  where the parameters have been sampled from the posterior distribution $\theta \sim p(\theta | y) $. 

We notice that the majority of the $\{y^{\text{rep}}\}$ are in the prediction interval which is not the case for the observed ground measurements $\{y\}$, It confirms what have previously be said when examining the residuals of a simple ordinary least squares regression.
- "*models based on the normality assumption of the output given the input may not be appropriate*"

The observed ground measurements contains more variability that is not captured by the model. Moreover when looking at the shape of the distribution, it is not a good fit for the observed ground measurements.

**Note** The prediction interval is defined as $[(\beta_0 + \beta_1 x_i) - 1.96 \sigma, (\beta_0 + \beta_1 x_i) + 1.96 \sigma]$. 

### Model 2

In [ ]:
def generate_replicates_2(df, thetas):
    replicates = []
    X, Y, J = df['log_aod'], df['log_pm25'], df['super_region']
    for theta in thetas:
        def sampler_2():
            b0, b1, log_sigma, log_tau_0, log_tau_1, *bjs = theta
            b0js = np.array(bjs[:7])
            b1js = np.array(bjs[7:])
            sigma = np.exp(log_sigma)
            tau_0 = np.exp(log_tau_0)
            tau_1 = np.exp(log_tau_1)
            return b0, b1, sigma, tau_0, tau_1, b0js ,b1js
            
        replicate = model_2(X, J, sampler_2, sample_from_prior=False)
        replicates.append(replicate)
        
    return np.array(replicates)

In [ ]:
n_replicates = 4

title = r"Posterior predictive distributions of replicated data $\{y^{rep}\}$ for Model 2"
plot_posterior_predictive_distribution(thetas_2[:n_replicates], df, generate_replicates_2, title)

Here, the model is slightly more complex than before and incorporates the effect of super-regions. For readability reasons, the graph does not show the prediction interval. However, the replicated data are a bit more spread out than the previous model. And seem quite close to the observed ground measurements.

The histogram is also more similar with a more skewed distribution on the right.

It confirms somethings told earlier:
- "[...] *super_region* variable is not independent of the *pm25* variable.*"

This model seems definitely better than the previous one.

In [ ]:
replicates_2 = generate_replicates_2(df, thetas_2[:4])
title = r"Posterior predictive distributions of replicated data $\{y^{rep}\}$ for Model 2 by super region"
plot_posterior_predictive_distribution_per_cluster(df, replicates_2, thetas_2[:4], title, 'super_region_name', 'super_region')

If we look at the replicate data according to the super-region, we can see that the model seems to be accurate for "SE-Asia/E-Asia/Oceania", "S-Asia"  while it struggles a bit more to capture well the variability of the other super-regions. 

### Model 3

In [ ]:
def generate_replicates_3(df, thetas):
    replicates = []
    X, Y, J = df['log_aod'], df['log_pm25'], df['cluster']
    for theta in thetas:
        def sampler_3():
            b0, b1, log_sigma, log_tau_0, log_tau_1, *bjs = theta
            b0js = np.array(bjs[:7])
            b1js = np.array(bjs[7:])
            sigma = np.exp(log_sigma)
            tau_0 = np.exp(log_tau_0)
            tau_1 = np.exp(log_tau_1)
            return b0, b1, sigma, tau_0, tau_1, b0js ,b1js
            
        replicate = model_3(X, J, sampler_3, sample_from_prior=False)
        replicates.append(replicate)
        
    return np.array(replicates)

In [ ]:
n_replicates = 4

title = r"Posterior predictive distributions of replicated data $\{y^{rep}\}$ for Model 3"
plot_posterior_predictive_distribution(thetas_3[:n_replicates], df, generate_replicates_3, title)

As the previous model this one is cluster based and make the assumption that close $\log pm25$ values are more likely to have a similar trend. 

This assumption seems to be confirmed by the histogram of the replicated data.

In [ ]:
replicates_3 = generate_replicates_3(df.sort_values('cluster'), thetas_3[:4])
title = r"Posterior predictive distributions of replicated data $\{y^{rep}\}$ for Model 3 by cluster"
plot_posterior_predictive_distribution_per_cluster(df.sort_values('cluster'), replicates_3, thetas_3, title, 'cluster_name', 'cluster')

Here the variability of the replicated data is more similar to the observed ground measurements, there are few outliers compare to the previous model.

However, visually, it is complicated to say which model is more accurate between the previous two models.

<div class="alert alert-success">
    
**Exercise**. Compare and discuss the posterior predictive distributions of test quantities of replicated data for Model 1, 2, and 3, using `mean` and `skew` as test quantities. Perform this check both at worldwide and cluster levels.
    
</div>

In [ ]:
n_replicates = 5000

### Model 1

In [ ]:
replicates_1 = generate_replicates_1(df, thetas_1[:n_replicates])

In [ ]:
plot_posterior_predictive_distributions_test_quantities(df, replicates_1)

Clearly the skewness of the replicated data is not the same as the observed ground measurements, the mean however is quite close.

### Model 2

In [ ]:
replicates_2 = generate_replicates_2(df, thetas_2[:n_replicates])

In [ ]:
plot_posterior_predictive_distributions_test_quantities(df, replicates_2)

The skewness of the replicated data is closer to the observed ground measurements than the previous model. The mean is equally close.

In [ ]:
plot_posterior_predictive_distributions_test_quantities_per_cluster(df, 'super_region', 'super_region_name', replicates_2)

By looking at the histograms of the mean and skewness of the replicated data, we can see that the model seems to perform well for the super-region "LatAm/Carib", "E-Eur/C-Eur/C-Asia" and "Sub-Saharan Afr", that contradicts what I said earlier, but at the time I only looked at a few replicates.

However for the other super-regions, the model does not capture the skewness of the observed ground measurements.

### Model 3

In [ ]:
replicates_3 = generate_replicates_3(df.sort_values('cluster'), thetas_3[:n_replicates])

In [ ]:
plot_posterior_predictive_distributions_test_quantities(df.sort_values('cluster'), replicates_3)

This histogram are more or less similar to the previous model, but the density of the replicated data is higher near the observed ground measurement skewness. Again, the mean is in the high density region.

In [ ]:
plot_posterior_predictive_distributions_test_quantities_per_cluster(df.sort_values('cluster'), 'cluster', 'cluster_name', replicates_3)

<div class="alert alert-success">
    
**Exercise**. Evaluate the posterior predictive performance of Model 1, 2, and 3.

</div>

In this section, I will evaluate the different models using an *Expected Log Predictive Density (EPLD)* estimate.

The models will be *trained* on a training set and evaluated on a test set. The training set is composed of 80% of the randomly selected data and the test set is composed of the remaining 20%.

This allows the models to be compared on unknown data in an unbiased way. If one wants to know the expected performance of the *best* model, one must use a validation set.

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
def log_posterior_predictive(xs, thetas, ll):
    if xs.shape[0] == 2:
        return np.mean([ll(xs[0], xs[1], *theta) for theta in thetas])
    else:
        return np.mean([ll(xs[0], xs[1], int(xs[2]), theta[0], theta[1], theta[2], theta[5:12], theta[12:]) for theta in thetas])

def epld(df, thetas, ll):
    return np.mean([log_posterior_predictive(x_i, thetas, ll) for x_i in df.values])

### Model 1

In [ ]:
df_1 = df_train.copy() # This change the df that uses log_posterior_model_1 (for optimization purposes)

In [ ]:
nwalkers = 50
ndim = 3
nsteps = 5000
burnin = 1000

init_states = res1.x + 1e-3 * np.random.randn(nwalkers, ndim)

df_1 = df_train.copy()

with Pool() as pool:
    sampler_1_eval = emcee.EnsembleSampler(nwalkers, ndim, log_posterior_model_1, pool=pool)
    sampler_1_eval.run_mcmc(init_states, nsteps, progress=True)

tau = sampler_1_eval.get_autocorr_time(quiet=True)
thin = int(np.max(tau))

thetas_1 = sampler_1_eval.get_chain(flat=True, discard=burnin, thin=thin)
thetas_1[:, 2] = np.exp(thetas_1[:, 2]) # sigma

In [ ]:
epld_1 = epld(df_test[['log_aod', 'log_pm25']], thetas_1[:500], log_likelihood_model_1)
print(f"EPLD for Model 1: {epld_1:.4f}")

### Model 2

In [ ]:
df_2 = df_train.copy() # This change the df that uses log_posterior_model_2 (for optimization purposes)

In [ ]:
nwalkers = 250
ndim = 19
nsteps = 25000
burnin = 5000

init_states = res2.x + 1e-3 * np.random.randn(nwalkers, ndim)

with Pool() as pool:
    sampler_2_eval = emcee.EnsembleSampler(nwalkers, ndim, log_posterior_model_2, pool=pool)
    sampler_2_eval.run_mcmc(init_states, nsteps, progress=True)

tau = sampler_2_eval.get_autocorr_time(quiet=True)
thin = int(np.max(tau))

thetas_2 = sampler_2_eval.get_chain(flat=True, discard=burnin, thin=thin)
thetas_2[:, 2] = np.exp(thetas_2[:, 2]) # sigma
thetas_2[:, 3] = np.exp(thetas_2[:, 3]) # tau_0
thetas_2[:, 4] = np.exp(thetas_2[:, 4]) # tau_1

In [ ]:
epld_2 = epld(df_test[['log_aod', 'log_pm25', 'super_region']], thetas_2[:500], log_likelihood_model_2)
print(f"EPLD for Model 2: {epld_2:.2f}")

### Model 3

In [ ]:
df_3 = df_train.copy() # This change the df that uses log_posterior_model_3 (for optimization purposes)

In [ ]:
nwalkers = 250
ndim = 19
nsteps = 25000
burnin = 5000

init_states = res3.x + 1e-3 * np.random.randn(nwalkers, ndim)

with Pool() as pool:
    sampler_3_eval = emcee.EnsembleSampler(nwalkers, ndim, log_posterior_model_3, pool=pool)
    sampler_3_eval.run_mcmc(init_states, nsteps, progress=True)

tau = sampler_3_eval.get_autocorr_time(quiet=True)
thin = int(np.max(tau))

thetas_3 = sampler_3_eval.get_chain(flat=True, discard=burnin, thin=thin)
thetas_3[:, 2] = np.exp(thetas_3[:, 2]) # sigma
thetas_3[:, 3] = np.exp(thetas_3[:, 3]) # tau_0
thetas_3[:, 4] = np.exp(thetas_3[:, 4]) # tau_1

In [ ]:
epld_3 = epld(df_test[['log_aod', 'log_pm25', 'cluster']], thetas_3[:500], log_likelihood_model_3)
print(f"EPLD for Model 3: {epld_3:.2f}")

It seems that the third model is the best one with respect to the *Expected log preditive density*. 

# Revise

<div class="alert alert-success">
    
**Exercise**. Explain and motivate what could be tried to further improve the probabilistic models consider in this study. (No implementation is required.)

</div>

The major issue is the skewness of the replicated data, one way to improve the probabilistic model would be to use a distribution that can capture the skewness of the data using a parameter. For exemple using a [*Skew Normal*](https://en.wikipedia.org/wiki/Skew_normal_distribution) instead of a *Normal* distribution may improve the model, as it could adapt the skewness of the distribution for each cluster with $\alpha_j$.

```
from scipy.stats import skewnorm
```

$y_{i, j} \sim \mathcal{SkewNormal}((\beta_0 + \beta_{0,j})\ +\ (\beta_1 + \beta_{1, j}) x_{i,j},\ \omega,\ \alpha_j)$

**Note**, the prior of $\alpha_j$ could be hierarchical like $\beta_{0,j}$ and $\beta_{1,j}$ to ensure that the skewness is not too different from the other clusters and stay plausible.